# Confronta de base de datos

In [1]:
import pandas as pd

thr_diff = 3

df = pd.read_pickle('data_clean/SEMEFO_BO_all.pkl')
df.head()

,ID,Numero_progresivo_transcrito,Nombre_completo_transcrito,Primer_apellido,Segundo_apellido,Nombres_propios,Fecha_transcrito,Fecha_estandar,Fecha_añomes,Fecha_diasemana,...,Procedencia_acta,Diagnostico_transcrito,Diagnostico_estandar,Diagnostico_extendido,Tipo_restos,Conocido_desconocido,Bitacora_ingresos,Pagina_PDF,Foja_transcrito,Observaciones
0,BO_1968_00001,NaN,acosta ortega teresa,acosta,ortega,teresa,1968-01-03,1968-01-03,1968 Enero,Miércoles,...,NaN,S-D,S-D,S-D,Cadáver,conocido,semefo_df_bo_1968,2,1,NaN
1,BO_1968_00002,NaN,avila de cuestas catalina,avila,de cuestas,catalina,1968-01-05,1968-01-05,1968 Enero,Viernes,...,NaN,S-D,S-D,S-D,Cadáver,conocido,semefo_df_bo_1968,2,1,NaN
2,BO_1968_00003,NaN,arzate paredes juan,arzate,paredes,juan,1968-01-07,1968-01-07,1968 Enero,Domingo,...,NaN,S-D,S-D,S-D,Cadáver,conocido,semefo_df_bo_1968,2,1,NaN
3,BO_1968_00004,NaN,alvarez martinez isaac,alvarez,martinez,isaac,1968-01-07,1968-01-07,1968 Enero,Domingo,...,NaN,S-D,S-D,S-D,Cadáver,conocido,semefo_df_bo_1968,2,1,NaN
4,BO_1968_00005,NaN,arellano viuda de campos ma.,arellano,viuda de campos,ma.,1968-01-07,1968-01-07,1968 Enero,Domingo,...,NaN,S-D,S-D,S-D,Cadáver,conocido,semefo_df_bo_1968,2,1,NaN


In [20]:
df['Procedencia_estandar'].nunique()

110

In [18]:
df.loc[df['Diagnostico_estandar'].str.contains('HPAF').fillna(False),['Diagnostico_transcrito','Diagnostico_estandar']].drop_duplicates()

,Diagnostico_transcrito,Diagnostico_estandar
32858,HPAFPT,HPAFPT
32873,HPAFPC,HPAFPC
32890,HPAFPA,HPAFPA
32894,HPAF,HPAF
32932,HPAFPV,HPAFPV
32974,HPPAFP,HPAF
32977,HPPAFPC,HPAFPC
33017,HPAFP,HPAF
33031,HsPAFPsC,HPAFPC
33095,HsPAFPsT,HPAFPT


## Ingenieria de grupos de ingreso por acta y fecha

In [2]:
grupos_interes = df['Procedencia_acta'].value_counts()
grupos_interes = grupos_interes[grupos_interes>1].index
grupos_interes

df['Grupo_ingreso'] = None
thr_diff=3

for id_grupo in grupos_interes:
    data = df.loc[df['Procedencia_acta']==id_grupo,['ID','Fecha_estandar']]
    data = data.sort_values(by='Fecha_estandar')
    
    ids = []
    nom_grupo=None
    for d in data.rolling(window=2):
        if len(d)>1:
            diff = (d.iloc[1,1]-d.iloc[0,1]).days
            if diff<=thr_diff:
                if nom_grupo==None: 
                    nom_grupo = id_grupo +' '+ d.iloc[0,1].strftime('%Y-%m-%d')
                ids += d['ID'].to_list()
    ids = list(set(ids))
    df.loc[df['ID'].isin(ids),'Grupo_ingreso']= nom_grupo


In [3]:
data = df.loc[df['Procedencia_acta'].isin(grupos_interes),['ID', 'Fecha_estandar', 'Procedencia_acta', 'Grupo_ingreso']] \
          .sort_values(by=['Procedencia_acta','Fecha_estandar']) #.dropna(subset='Grupo_ingreso') \
data.iloc[50:100]

,ID,Fecha_estandar,Procedencia_acta,Grupo_ingreso
33823,BO_1974_00987,1974-11-28,10A/1483,None
54195,BO_1977_01860,1977-06-06,10A/1483,None
52516,BO_1977_00181,1977-06-17,10A/1483,None
33142,BO_1974_00306,1974-12-09,10A/1489,None
92702,BO_1982_03352,1982-05-12,10A/1489,None
56691,BO_1977_04356,1977-05-07,10A/1498,None
56673,BO_1977_04338,1977-06-07,10A/1498,None
55518,BO_1977_03183,1977-06-16,10A/1570,None
83766,BO_1981_02045,1981-05-25,10A/1570,None
69199,BO_1979_02558,1979-10-01,10A/1593,None


## Ingenieria por enfrentamientos

In [5]:
from datetime import timedelta

fechas = pd.read_csv('data_raw/Enfrentamientos_CDMX - 20230214_Datos.csv')
fechas = pd.to_datetime( fechas['Fecha'] ).dropna().sort_values().unique()
fechas

dic_fechas = {}
f_range = list(range(-1*thr_diff, thr_diff+1))
for f in fechas:
    f_str = 'Enfrentamiento ' +f.strftime('%Y-%m-%d')
    dic_fechas.update( {f+timedelta(days=i):f_str for i in f_range} )
dic_fechas

df['Enfrentamientos_fecha'] = df['Fecha_estandar'].map(dic_fechas)
df['Enfrentamientos_fecha'].value_counts()

Enfrentamientos_fecha
Enfrentamiento 1977-12-17    202
Enfrentamiento 1981-01-22    173
Enfrentamiento 1977-01-19    167
Enfrentamiento 1979-10-13    166
Enfrentamiento 1977-01-26    161
Enfrentamiento 1978-09-15    154
Enfrentamiento 1978-08-18    148
Enfrentamiento 1977-05-18    148
Enfrentamiento 1975-12-13    147
Enfrentamiento 1979-10-04    146
Enfrentamiento 1978-01-05    145
Enfrentamiento 1974-12-10    143
Enfrentamiento 1980-04-23    141
Enfrentamiento 1977-04-14    137
Enfrentamiento 1973-12-19    136
Enfrentamiento 1979-08-29    134
Enfrentamiento 1976-08-28    134
Enfrentamiento 1977-09-03    131
Enfrentamiento 1973-12-01    131
Enfrentamiento 1975-04-25    130
Enfrentamiento 1975-08-22    127
Enfrentamiento 1976-06-09    126
Enfrentamiento 1976-08-11    124
Enfrentamiento 1979-09-07    123
Enfrentamiento 1968-10-02    120
Enfrentamiento 1979-03-30    119
Enfrentamiento 1977-06-24    117
Enfrentamiento 1974-10-11    117
Enfrentamiento 1976-06-19    116
Enfrentamiento 1974-0

## Guardar

In [6]:
df.to_csv('data_clean/SEMEFO_BO_all_extendido.csv', index=False)
df.to_pickle('data_clean/SEMEFO_BO_all_extendido.pkl')

In [18]:
df.columns

Index(['ID', 'Numero_progresivo_transcrito', 'Nombre_completo_transcrito',
       'Primer_apellido', 'Segundo_apellido', 'Nombres_propios',
       'Fecha_transcrito', 'Fecha_estandar', 'Fecha_añomes', 'Fecha_diasemana',
       'Sexo', 'Edad_transcrito', 'Edad_estandar', 'Edad_rango',
       'Expediente_SEMEFO_transcrito', 'Procedencia_transcrito',
       'Procedencia_estandar', 'Procedencia_direccion', 'Procedencia_alcaldia',
       'Numero_acta_transcrito', 'Procedencia_acta', 'Diagnostico_transcrito',
       'Diagnostico_estandar', 'Diagnostico_extendido', 'Tipo_restos',
       'Conocido_desconocido', 'Bitacora_ingresos', 'Pagina_PDF',
       'Foja_transcrito', 'Observaciones', 'Grupo_ingreso',
       'Enfrentamientos_fecha'],
      dtype='object')

In [20]:
df.loc[(df['Diagnostico_estandar'].str.contains('HPAF')) & (df['Enfrentamientos_fecha'].notna()) & (df['Grupo_ingreso'].notna()),
       ['ID', 'Nombre_completo_transcrito', 'Fecha_transcrito', 'Edad_transcrito', 'Procedencia_acta', 'Diagnostico_transcrito']]


,ID,Nombre_completo_transcrito,Fecha_transcrito,Edad_transcrito,Procedencia_acta,Diagnostico_transcrito
33733,BO_1974_00897,correa hernandez antelmo,1974-10-09,70,27A/745,HSPAFPsA
33734,BO_1974_00898,castillo perez ivan,1974-10-09,35,27A/745,HsPAFPsCT
39005,BO_1975_00438,barcenas orduna alberto,1975-04-26,45,TP-1/222,HsPAFPs
39322,BO_1975_00755,camahco cruz m francisco,1975-04-26,28,TP-1/222,HsPAFPs
39461,BO_1975_00894,calderon salgado raul,1975-08-19,20,AO-1/1304,HsPAFPsT
39881,BO_1975_01314,espinoza olascoaga miguel,1975-04-26,30,TP-1/222,HsPAFPsC
40269,BO_1975_01702,gonzalez miranda sabino,1975-04-21,26,C-1/640,HsPAFPsT
40270,BO_1975_01703,gonzalez miranda eugenio,1975-04-21,24,C-1/640,HPAFPC
40298,BO_1975_01731,garza arrieta enrique,1975-04-26,55,TP-1/222,HsPAFPsTA
40462,BO_1975_01895,garcia raul,1975-08-18,25,MH-CR/885,HPAFPT
